In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 7 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 calibration.
# - Centre search on the NEW actual incumbent.
# - Use local + moderate-wide candidate pools.
# - Keep some global coverage as a diagnostic.
# - Compare EI, posterior mean and UCB.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------
#
# Week 9 selected:
# [0.16522977, 0.22799513, 0.58728299,
#  0.25148122, 0.27764608, 0.67443723]
#
# Predicted:
# mean ≈ 2.779768
# std  ≈ 0.078423
#
# Actual:
# 3.0575674596116973
# ------------------------------------------------------------

week9_pred_mean = 2.7797683614352584
week9_pred_std = 0.07842338624398089
week9_actual = 3.0575674596116973

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(6) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------
#
# Week 9 produced a new best from moderate exploration.
# Keep the search centred on the new incumbent but retain
# controlled wider exploration.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(160000, 6)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(120000, 6)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(120000, 6)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from incumbent
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 13. Domain-boundary check
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.01
):

    status = []

    for j in range(len(x)):

        if x[j] <= tol:
            status.append(
                f"x{j+1}~0"
            )

        elif x[j] >= 1.0 - tol:
            status.append(
                f"x{j+1}~1"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )

DATA
X shape: (39, 6)
Y shape: (39,)

Current best:
[0.16523  0.227995 0.587283 0.251481 0.277646 0.674437] -> 3.0575674596116973

Y range:
min = 0.0027014650245082332
max = 3.0575674596116973
std = 0.9548416137214286

WEEK 9 CALIBRATION CHECK
Predicted mean: 2.7797683614352584
Predicted std : 0.07842338624398089
Actual        : 3.0575674596116973

Prediction error:
0.2777990981764389

Error / predicted std:
3.5422991977442235


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
0.648**2 * Matern(length_scale=[0.466, 0.584, 2, 0.304, 0.204, 0.383], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[0.46557078 0.58434572 2.         0.30416466 0.20446378 0.38276953]

Normalised inverse-lengthscale sensitivity:
[0.14177294 0.11295597 0.03300267 0.21700529 0.32282166 0.17244147]

CANDIDATE SCALES
Local widths:
[0.08       0.08       0.08       0.06083293 0.04089276 0.07655391]

Wide widths:
[0.15       0.15       0.15       0.12166586 0.08178551 0.15      ]

Candidates after duplicate filtering:
400000

PRIMARY EI
candidate = [0.18357378 0.17492569 0.69059107 0.26728517 0.29651548 0.68069354]
mean = 3.1063997281066196
std = 0.07173493359439305
EI = 0.05942032621837785

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.18357378 0.17492569 0.69059107 0.26728517 0.29651548 0.68069354] 
 mean = 3.1064 
 std = 0.071735 
 EI = 0.05942033 

xi = 9.548416e-03 
 candidate = [0.18357378 0.17492569 0.69059107 0.26728517 0.29651548 0.6806935

In [2]:
# ============================================================
# FINAL FUNCTION 7 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 produced a large positive calibration surprise
# (+3.54 predictive standard deviations) and a new best.
#
# After refitting, primary EI, highest posterior mean,
# and UCB beta = 0.05, 0.1, 0.25 and 0.5 all identify
# exactly the same candidate.
#
# This is strong acquisition agreement based on both
# exploitation and moderate exploration.
#
# beta = 1.0 is rejected because it moves to a lower-mean,
# higher-uncertainty alternative.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week10_candidate = candidates[final_idx]

print("Week 10 Function 7 candidate:")
print(week10_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 7 candidate:
[0.18357378 0.17492569 0.69059107 0.26728517 0.29651548 0.68069354]

Predicted mean:
3.1063997281066196

Predicted std:
0.07173493359439305

UCB:
3.142267194903816

Distance from current best:
0.12029287739875655

Portal format:
0.183574-0.174926-0.690591-0.267285-0.296515-0.680694
